# Clase 5 · Transformar con Spark: de 20 millones de viajes crudos a un resumen confiable

Hasta la clase 4 consultaste datos que ya venían procesados. En esta clase empiezas a construir la lógica que los produce. El caso es el de todo el curso: ¿la lluvia cambia los viajes en NYC? Hoy trabajas con enero de 2024: 19 663 930 viajes de vehículos de alquiler con conductor (FHVHV, es decir Uber, Lyft y similares), las 265 zonas de la ciudad y el clima diario de Open-Meteo.

El notebook sigue un orden fijo. Primero preparas el entorno y arrancas Spark. Después comprendes cómo Spark reparte el trabajo y haces la misma pregunta con pandas, con Spark y con Snowflake para comparar tiempos y memoria con datos reales. Luego ves por qué algunas operaciones se paralelizan y otras no, y escribes tus propias funciones de bronze, silver y gold, cada una con una prueba que te dice al instante si está bien. Al final repites a mano, desde la terminal, lo que el notebook hizo, y verás que `make` solo le pone nombre a esos comandos.

Cómo se usa: ejecuta las celdas en orden, una por una, con Shift+Enter. Cada celda de código imprime lo que mide. Las cifras de este texto no existen a propósito: cada vez que haría falta una, el texto dice «lo verás al ejecutar» y el número lo produce la celda en tu propio entorno. Solo puede haber un proceso de Spark a la vez en esta máquina, así que no abras este notebook dos veces ni ejecutes `make test` mientras Spark siga encendido (más adelante te digo cuándo apagarlo).

## 0. Preparación: el entorno correcto, tu bucket y Spark encendido

Antes de tocar datos conviene comprobar tres cosas: que este notebook corre con el Python del proyecto, que sabes cuál es tu bucket y que el archivo de enero está en el disco de esta máquina. Las dos primeras evitan la mayoría de los errores de la clase; la tercera evita descargar 451 MiB cada vez.

### 0.1 Verificar el kernel

El kernel es el programa Python que ejecuta las celdas. Si eliges uno equivocado, las librerías no estarán instaladas. La celda siguiente imprime qué Python estás usando y si Java está disponible, porque Spark corre sobre la máquina virtual de Java.

In [ ]:
# Java: el kernel del notebook no lee ~/.bashrc, así que puede no ver el Java que instaló make setup
# en ~/.local/opt/java17. Si no está en el PATH, lo buscamos ahí y definimos JAVA_HOME antes de usar Spark.
import os, shutil
from pathlib import Path
if shutil.which("java") is None:
    for candidato in [os.environ.get("JAVA_HOME"), str(Path.home() / ".local/opt/java17")]:
        if candidato and (Path(candidato) / "bin/java").exists():
            os.environ["JAVA_HOME"] = candidato
            os.environ["PATH"] = f"{candidato}/bin:" + os.environ["PATH"]
            break

import json, os, re, shutil, subprocess, sys, time, contextlib, gc
from io import StringIO
from pathlib import Path

print("Python:", sys.version.split()[0])
print("Ejecutable:", sys.executable)
if ".venv" not in sys.executable:
    print("AVISO: este kernel no es el del proyecto (la ruta no contiene .venv). "
          "Elige el kernel «Python (bigdata)» arriba a la derecha y reinicia el notebook.")

try:
    java = subprocess.run(["java", "-version"], capture_output=True, text=True)
    print("Java:", (java.stderr or java.stdout).splitlines()[0])
except FileNotFoundError:
    print("AVISO: no se encontró Java en el PATH; Spark no podrá arrancar.")

import pandas, pyarrow, pyspark
print("pandas", pandas.__version__, "| pyarrow", pyarrow.__version__, "| pyspark", pyspark.__version__)

### 0.2 Tu bucket, armado desde tu cuenta de AWS

Cada estudiante tiene su propio lake en S3. Su nombre no se escribe a mano: se arma con un patrón fijo, `<prefijo>-lake-<número de cuenta>-<región>`, igual que hace la función `lake_bucket` de `scripts/lib.sh` cuando corres un `make`. El número de cuenta se lo preguntamos a AWS con las credenciales del Code Editor (es lo mismo que `aws sts get-caller-identity`). La celda lo arma, lo imprime y comprueba que existe y que puedes leerlo.

In [ ]:
import boto3
from botocore.exceptions import ClientError

REGION = "us-east-2"

# Raíz del repositorio: la carpeta que contiene pyproject.toml, subiendo desde donde está este notebook.
RAIZ = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()),
            Path.home() / "bigdata")
sys.path.insert(0, str(RAIZ / "src"))   # así se puede hacer «from pipeline import ...»

# El prefijo viene del archivo .env del repo (lo escribe «make setup»); si no existe, es «bigdata».
prefijo = os.environ.get("PREFIX")
if not prefijo and (RAIZ / ".env").exists():
    m = re.search(r"^PREFIX=(.+)$", (RAIZ / ".env").read_text(), re.M)
    prefijo = m.group(1).strip() if m else None
prefijo = prefijo or "bigdata"

cuenta = boto3.client("sts", region_name=REGION).get_caller_identity()["Account"]
BUCKET = f"{prefijo}-lake-{cuenta}-{REGION}"
print("Cuenta AWS :", cuenta)
print("Tu bucket  :", BUCKET)

s3 = boto3.client("s3", region_name=REGION)
try:
    s3.head_bucket(Bucket=BUCKET)
    print("Acceso al bucket: correcto")
except ClientError as e:
    print("AVISO: no se pudo leer el bucket:", e.response["Error"]["Code"],
          "- se usará el bucket público del curso para el archivo de enero.")

### 0.3 El archivo de enero, una sola vez, en el disco local

Los tres motores que vas a comparar leen el mismo archivo Parquet. Pandas y Spark lo leen desde el disco de esta máquina, así que primero hay que traerlo desde tu bucket (`bronze/fhvhv/mes=2024-01/`). Se guarda fuera del repositorio, en la carpeta `~/datos_clase05`, para que nunca llegue a un commit por accidente. Si la carpeta ya tiene el archivo, la celda no vuelve a descargarlo. Si tu bucket no lo tiene, usa la copia del bucket público del curso. Con las zonas pasa lo mismo, y si tampoco están en tu bucket se bajan del sitio de la TLC, que es de donde las toma `bronze.py`.

In [ ]:
from botocore import UNSIGNED
from botocore.config import Config
import urllib.request

DATOS = Path(os.environ.get("CLASE05_DATOS", Path.home() / "datos_clase05"))
CLAVE_ENERO = "bronze/fhvhv/mes=2024-01/fhvhv_tripdata_2024-01.parquet"
CLAVE_ZONAS = "bronze/taxi_zones/taxi_zone_lookup.csv"
RUTA_DIR_ENERO = DATOS / "bronze" / "fhvhv" / "mes=2024-01"
RUTA_ENERO = RUTA_DIR_ENERO / "fhvhv_tripdata_2024-01.parquet"
RUTA_ZONAS = DATOS / "bronze" / "taxi_zones" / "taxi_zone_lookup.csv"
MB = 1024 * 1024


def traer(clave, destino, url_respaldo=None):
    """Descarga s3://BUCKET/clave a destino si todavía no existe (primero tu bucket, luego el público)."""
    if destino.exists():
        print(f"Ya existe, no se descarga: {destino}  ({destino.stat().st_size / MB:,.1f} MiB)")
        return
    destino.parent.mkdir(parents=True, exist_ok=True)
    parcial = destino.with_name(destino.name + ".parcial")   # evita dejar un archivo a medias
    t0 = time.perf_counter()
    try:
        s3.download_file(BUCKET, clave, str(parcial))
        origen = f"s3://{BUCKET}/{clave}"
    except ClientError:
        if url_respaldo:
            urllib.request.urlretrieve(url_respaldo, parcial)
            origen = url_respaldo
        else:
            publico = boto3.client("s3", region_name=REGION, config=Config(signature_version=UNSIGNED))
            publico.download_file("bigdata-udem-2026-publico", "paquete/" + clave, str(parcial))
            origen = f"s3://bigdata-udem-2026-publico/paquete/{clave}"
    parcial.rename(destino)
    print(f"Descargado desde {origen}\n  -> {destino}  ({destino.stat().st_size / MB:,.1f} MiB en {time.perf_counter() - t0:.1f} s)")


from pipeline import bronze as bronze_pipeline   # trae URL_ZONAS, url_clima y descargar
traer(CLAVE_ENERO, RUTA_ENERO)
traer(CLAVE_ZONAS, RUTA_ZONAS, url_respaldo=bronze_pipeline.URL_ZONAS)

### 0.4 Dos utilidades para medir

En esta clase medirás tiempo y memoria muchas veces. Para no repetir código, la celda siguiente define un cronómetro que imprime lo que tardó un bloque y lo guarda en el diccionario `TIEMPOS`, y dos funciones que leen cuánta memoria usa el proceso de este notebook. Fíjate en que la memoria que se mide es la del proceso de Python; la máquina virtual de Java que usa Spark es otro proceso aparte y se ve por separado.

In [ ]:
TIEMPOS = {}


@contextlib.contextmanager
def cronometro(etiqueta):
    """Mide un bloque, lo imprime y lo guarda en TIEMPOS."""
    t0 = time.perf_counter()
    yield
    TIEMPOS[etiqueta] = time.perf_counter() - t0
    print(f"[{etiqueta}] {TIEMPOS[etiqueta]:.2f} s")


def rss_mb():
    """Memoria residente actual del proceso del notebook (MB). En macOS no está disponible."""
    try:
        with open("/proc/self/status") as f:
            for linea in f:
                if linea.startswith("VmRSS:"):
                    return int(linea.split()[1]) / 1024
    except FileNotFoundError:
        pass
    return float("nan")


def pico_mb():
    """Máximo de memoria residente que ha alcanzado el proceso del notebook desde que arrancó (MB)."""
    import resource
    pico = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    return pico / MB if sys.platform == "darwin" else pico / 1024


def fmt(n):
    """19663930 -> '19 663 930'"""
    return f"{n:,}".replace(",", " ")


def memoria_equipo_gb():
    return os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1024**3


print(f"Memoria total de esta máquina: {memoria_equipo_gb():.1f} GB")
print(f"Memoria del proceso del notebook ahora: {rss_mb():,.0f} MB")

### 0.5 Arrancar Spark con la misma configuración del pipeline

No vamos a inventar una configuración aparte para el notebook: usamos la misma función `get_spark("local")` de `src/pipeline/engine.py` que usa el pipeline. Crea una sesión `local[*]`, que significa que el driver y los ejecutores viven en un solo proceso y que Spark usa tantos hilos como núcleos tenga la máquina. El driver tiene 2 GB de memoria, la zona horaria de la sesión queda en UTC y los reemplazos de particiones son dinámicos. Un solo proceso Spark a la vez: si ya lo habías arrancado en otro notebook o en una terminal, ciérralo antes.

Después de arrancar, la celda imprime `defaultParallelism`, el número de tareas que Spark puede ejecutar al mismo tiempo, y el enlace a la Spark UI, la página donde Spark muestra cada trabajo, etapa y tarea mientras corre. La usarás en las secciones 3 y 4.

In [ ]:
# La Spark UI se ve a través del proxy del Code Editor, que agrega el prefijo /codeeditor/default/ports/4040.
# Spark necesita saberlo para armar sus enlaces, estilos e imágenes con ese prefijo; si no, la página
# carga sin diseño. Se indica antes de arrancar Spark, porque se lee al encender la JVM.
import os
os.environ["PYSPARK_SUBMIT_ARGS"] = "--conf spark.ui.proxyBase=/codeeditor/default/ports/4040 pyspark-shell"

from pyspark.sql import functions as F, DataFrame, Window
from pipeline.engine import get_spark

with cronometro("arrancar Spark"):
    spark = get_spark("local")

print("Versión de Spark :", spark.version)
print("Master           :", spark.sparkContext.master)
print("Memoria del driver:", spark.sparkContext.getConf().get("spark.driver.memory"))
print("defaultParallelism (tareas simultáneas):", spark.sparkContext.defaultParallelism)


def url_spark_ui():
    """Enlace a la Spark UI. Dentro del Code Editor pasa por el proxy del Space, igual que Airflow."""
    local = spark.sparkContext.uiWebUrl            # p. ej. http://10.0.0.5:4040
    puerto = local.rsplit(":", 1)[-1].strip("/")
    meta = Path("/opt/ml/metadata/resource-metadata.json")
    if meta.exists():
        try:
            datos = json.loads(meta.read_text())
            r = boto3.client("sagemaker", region_name=REGION).describe_space(
                DomainId=datos["DomainId"], SpaceName=datos["SpaceName"])
            base = re.match(r"https://[^/]+", r["Url"]).group(0)
            return f"{base}/codeeditor/default/ports/{puerto}/jobs/"   # /jobs/ directo: la raíz redirige mal detrás del proxy
        except Exception as e:                      # sin permiso o fuera del Code Editor
            print("(no se pudo consultar el Space:", type(e).__name__, ")")
    return f"{local}   (en el Code Editor: https://<id-del-space>.studio.{REGION}.sagemaker.aws/codeeditor/default/ports/{puerto}/)"


print("Spark UI:", url_spark_ui())

**Qué observar.** `defaultParallelism` es el número de hilos que Spark usa como «ejecutores» en esta máquina; esa cifra manda cuántas tareas avanzan a la vez. Abre el enlace de la Spark UI en otra pestaña y déjalo abierto: la pestaña *Jobs* está vacía porque Spark todavía no ha hecho nada, y se irá llenando conforme ejecutes celdas. La UI solo existe mientras la sesión de Spark está viva.

## 1. Spark divide los datos en particiones y las procesa en paralelo

**El problema.** Pandas carga la tabla completa en la memoria de un solo proceso. Mientras la tabla quepa, es rapidísimo; cuando no cabe, el proceso se cae. Con 19.6 millones de filas ya se nota, y con los 239.5 millones de viajes de 2024 el problema es de otra escala. **La idea de ingeniería** es no cargar todo: partir los datos en trozos independientes, las *particiones*, y repartir el trabajo entre varios procesadores. **Lo que resuelve Spark** es exactamente eso: planificar el trabajo, repartirlo y juntar los resultados. Lo que no resuelve es la lógica: si tu transformación obliga a juntar todo en un solo lugar, Spark también se queda esperando.

### Los cuatro nombres de la arquitectura

El *driver* es el proceso que lee tu código, construye el plan de ejecución y reparte el trabajo. Los *ejecutores* son los procesos (o hilos) que hacen el trabajo sobre los datos. Una *partición* es un trozo de los datos que se procesa de una sola vez. Una *tarea* es la unidad de trabajo que se le entrega a un ejecutor: una tarea por partición y por etapa. En un clúster real los ejecutores son máquinas distintas; en el Code Editor, Spark es un clúster en miniatura: un driver y varios hilos dentro de la misma máquina. La lógica es idéntica, solo cambia dónde viven los ejecutores.

La celda siguiente lo comprueba con tus datos: lee el archivo de enero sin cargarlo, cuenta cuántas particiones forma Spark y cuántas filas cae en cada una.

In [ ]:
with cronometro("leer metadatos del archivo (sin cargar filas)"):
    viajes_sp = spark.read.parquet(str(RUTA_DIR_ENERO))

n_part = viajes_sp.rdd.getNumPartitions()
print("Particiones que Spark formó para el archivo de enero:", n_part)
print("Tamaño máximo de partición al leer archivos (bytes):", spark.conf.get("spark.sql.files.maxPartitionBytes"))
print("Tareas que pueden correr a la vez:", spark.sparkContext.defaultParallelism)

In [ ]:
with cronometro("filas por partición"):
    por_particion = (viajes_sp
                     .groupBy(F.spark_partition_id().alias("particion"))
                     .count()
                     .orderBy("particion")
                     .toPandas())

print(por_particion.to_string(index=False))
print("Suma de las particiones:", fmt(int(por_particion["count"].sum())), "filas")

**Qué observar.** Cada fila de la tabla es una partición, y cada una será una tarea. El número de particiones no lo decidiste tú: Spark corta el archivo Parquet en pedazos del tamaño de `spark.sql.files.maxPartitionBytes` (o menos) siguiendo los grupos de filas internos del archivo. Compara ese número con `defaultParallelism`: si hay más particiones que hilos, las tareas esperan turno en grupos; si hay menos, algunos hilos se quedan sin trabajo. La suma de las filas por partición debe dar las 19 663 930 filas del archivo (la celda las imprime para que lo compruebes). Si abres la Spark UI en la pestaña *Jobs*, verás el trabajo que acabas de lanzar y, dentro de él, una tarea por partición.

## 2. La misma pregunta en pandas, en Spark y en Snowflake

Para que la comparación sea justa, los tres motores responden **exactamente la misma pregunta** sobre **el mismo archivo** de enero: ¿cuántos viajes hubo y cuánto se facturó en tarifa base (`base_passenger_fare`) por zona de recogida (`PULocationID`) y por día de recogida? Pandas y Spark leen el archivo desde el disco local, que viene de tu bucket. Snowflake lo lee desde el bucket público del curso, que contiene el mismo archivo de enero, sin credenciales. Al final comprobarás que los tres devuelven los mismos totales; si no coincidieran, la comparación de tiempos no tendría sentido.

Los tiempos que verás dependen de esta máquina. Por eso el texto no trae cifras: las produce cada celda.

In [ ]:
INICIO, FIN = "2024-01-01", "2024-02-01"            # enero, fin exclusivo
COLUMNAS_PREGUNTA = ["pickup_datetime", "PULocationID", "base_passenger_fare"]
RESULTADOS = {}                                      # aquí queda lo que responde cada motor

### 2.1 Pandas: leer solo lo necesario

El archivo tiene 24 columnas, pero la pregunta usa tres. Parquet guarda los datos por columna, así que se pueden leer solo esas tres sin tocar las demás. Eso lo hace `pyarrow` con el argumento `columns=`. La celda mide cada paso por separado: leer, convertir a un DataFrame de pandas y calcular. También guarda el pico de memoria del proceso, que es el dato que más importa con pandas.

In [ ]:
import pyarrow.parquet as pq
import pandas as pd

rss_antes, pico_antes = rss_mb(), pico_mb()

with cronometro("pandas 1/3: leer 3 columnas con pyarrow"):
    tabla = pq.read_table(RUTA_ENERO, columns=COLUMNAS_PREGUNTA)
with cronometro("pandas 2/3: convertir a DataFrame"):
    viajes_pd = tabla.to_pandas()
mb_df = viajes_pd.memory_usage(deep=True).sum() / MB
with cronometro("pandas 3/3: filtrar, agrupar y sumar"):
    enero = viajes_pd[(viajes_pd["pickup_datetime"] >= INICIO) & (viajes_pd["pickup_datetime"] < FIN)]
    res_pandas = (enero.assign(fecha=enero["pickup_datetime"].dt.normalize())
                  .groupby(["PULocationID", "fecha"], as_index=False)
                  .agg(viajes=("base_passenger_fare", "size"), ingreso=("base_passenger_fare", "sum")))

pico_despues = pico_mb()
RESULTADOS["pandas"] = dict(
    df=res_pandas.rename(columns={"PULocationID": "pu_location_id"}),
    segundos=sum(TIEMPOS[k] for k in TIEMPOS if k.startswith("pandas ")),
    pico_mb=pico_despues,
)
print(f"\nFilas leídas: {fmt(len(viajes_pd))} | filas en enero: {fmt(len(enero))} | grupos zona-día: {fmt(len(res_pandas))}")
print(f"DataFrame con 3 columnas: {mb_df:,.0f} MB")
print(f"Memoria del proceso: antes {rss_antes:,.0f} MB | pico del proceso hasta aquí {pico_despues:,.0f} MB (antes de la celda: {pico_antes:,.0f} MB)")
res_pandas.head()

**Qué observar.** El DataFrame con solo tres columnas ocupa una cantidad de memoria que ya se ve en la celda, y el *pico* del proceso es bastante mayor que eso, porque pandas hace copias intermedias al filtrar, al crear la columna `fecha` y al agrupar. El tiempo de pandas suele ser corto: cuando los datos caben en memoria, pandas es difícil de superar. La pregunta de ingeniería es qué pasa cuando ya no caben.

### 2.2 Qué pasaría con todas las columnas o con el año completo

Para responderlo sin arriesgar el kernel, la celda siguiente lee **un solo grupo de filas** del archivo con las 24 columnas, mide cuánta memoria ocupa en pandas y extrapola al archivo completo y al año. La extrapolación es una estimación aritmética (filas totales por memoria por fila), no una medición; el grupo de filas sí se mide. Las columnas de texto (`hvfhs_license_num`, `dispatching_base_num` y las banderas) son las que más pesan en pandas, porque cada valor se convierte en un objeto de Python.

In [ ]:
del tabla, viajes_pd, enero
gc.collect()

archivo = pq.ParquetFile(RUTA_ENERO)
grupo = archivo.read_row_group(0)                       # todas las columnas, un solo grupo de filas
mb_grupo = grupo.to_pandas().memory_usage(deep=True).sum() / MB
filas_archivo = archivo.metadata.num_rows
est_mes = mb_grupo * filas_archivo / grupo.num_rows
FILAS_ANIO = 239_500_000                                 # viajes FHVHV de 2024 (dato del caso del curso)
est_anio = est_mes * FILAS_ANIO / filas_archivo

print(f"Un grupo de {fmt(grupo.num_rows)} filas con las {grupo.num_columns} columnas ocupa {mb_grupo:,.0f} MB en pandas (medido).")
print(f"Estimación para enero completo con todas las columnas: {est_mes / 1024:,.1f} GB")
print(f"Estimación para el año (239.5 M de viajes): {est_anio / 1024:,.1f} GB")
print(f"Memoria total de esta máquina: {memoria_equipo_gb():.1f} GB")
del grupo
gc.collect()

**Qué observar.** Compara las dos estimaciones con la memoria total de la máquina. Eso es lo que significa «pandas carga todo en una máquina»: para el mes con todas las columnas ya hay que pensar, y para el año completo no hay forma de que quepa en esta máquina. Dos salidas posibles: una máquina mucho más grande (escalar hacia arriba, con un costo que crece) o repartir el trabajo (Spark). Y aun así hay una tercera defensa, la que acabas de usar: no leer lo que no necesitas.

### 2.3 Spark: la misma pregunta con SQL y con la API de DataFrames

Spark permite escribir la consulta de dos maneras equivalentes. La primera registra el DataFrame como una vista temporal y usa SQL; la segunda encadena métodos del DataFrame. Las dos producen el mismo plan de ejecución. El archivo se lee completo en esta celda, y como el sistema operativo guarda en caché lo que ya leyó, la segunda ejecución puede ser más rápida que la primera: es un efecto de la caché de archivos del sistema, no de Spark.

In [ ]:
viajes_sp.createOrReplaceTempView("viajes")

rss_antes = rss_mb()
with cronometro("spark SQL"):
    res_sql = spark.sql(f"""
        SELECT PULocationID AS pu_location_id,
               to_date(pickup_datetime) AS fecha,
               COUNT(*) AS viajes,
               SUM(base_passenger_fare) AS ingreso
        FROM viajes
        WHERE pickup_datetime >= '{INICIO}' AND pickup_datetime < '{FIN}'
        GROUP BY PULocationID, to_date(pickup_datetime)
    """).toPandas()

with cronometro("spark DataFrame"):
    res_df = (viajes_sp
              .where((F.col("pickup_datetime") >= INICIO) & (F.col("pickup_datetime") < FIN))
              .groupBy(F.col("PULocationID").alias("pu_location_id"),
                       F.to_date("pickup_datetime").alias("fecha"))
              .agg(F.count(F.lit(1)).alias("viajes"), F.sum("base_passenger_fare").alias("ingreso"))
              .toPandas())

RESULTADOS["spark"] = dict(df=res_sql, segundos=TIEMPOS["spark SQL"], pico_mb=pico_mb())
print(f"\nGrupos zona-día: SQL {fmt(len(res_sql))} | DataFrame {fmt(len(res_df))}")
print(f"Memoria del proceso del notebook (Python): {rss_antes:,.0f} MB antes, {rss_mb():,.0f} MB después")
print("La memoria de Spark vive en el proceso Java (driver de 2 GB), no en este.")
res_sql.head()

**Qué observar.** Primero, que el DataFrame y el SQL dan el mismo número de grupos. Segundo, que la memoria del proceso de Python casi no se movió: Spark no pasó los 19.6 millones de filas por Python, solo devolvió el resultado final (unos miles de filas) con `toPandas()`. Tercero, que Spark nunca tuvo todo el archivo en memoria a la vez: leyó una partición por tarea, la agregó y siguió. En la Spark UI, pestaña *Jobs*, verás los dos trabajos que acabas de ejecutar.

### 2.4 Snowflake, desde el notebook

Snowflake puede leer un archivo Parquet que está en S3 sin cargarlo antes a una tabla: se crea un *stage* (una referencia a una ubicación de archivos) y se consulta con SQL. Aquí el stage es **temporal** (desaparece al cerrar la sesión) y apunta al bucket público del curso, que se puede leer sin credenciales. La conexión usa la llave que ya configuraste: `snowflake.connector.connect(connection_name="default")` lee `~/.snowflake/connections.toml`, el mismo archivo que usa `snow`.

Dos detalles de medición que importan. Uno: Snowflake guarda el resultado de cada consulta y, si repites la misma, lo devuelve sin ejecutar nada; para medir de verdad se desactiva con `USE_CACHED_RESULT = FALSE`. Dos: si el warehouse estaba suspendido, la primera consulta paga el arranque; por eso se hace antes una consulta trivial y su tiempo se mide aparte. El tiempo de la consulta incluye leer 451 MiB desde S3, interpretar el Parquet y bajar el resultado al notebook.

Para leer el Parquet se usa `USE_VECTORIZED_SCANNER = TRUE` en el formato del stage. Sin esa opción Snowflake interpreta cada fila como un valor semiestructurado (`VARIANT`) y la misma consulta puede tardar muchas veces más. Si tienes curiosidad, quita esa opción del formato, repite la celda y compara el tiempo.

In [ ]:
import snowflake.connector

SF_BASE, SF_ESQUEMA = "CURSO", "CLASE5"      # CURSO ya existe desde las clases anteriores
BUCKET_PUBLICO = "s3://bigdata-udem-2026-publico/paquete/bronze/fhvhv/mes=2024-01/"

con = snowflake.connector.connect(connection_name="default")
cur = con.cursor()

def sf(sql):
    """Ejecuta una sentencia y devuelve el cursor."""
    return cur.execute(sql)

# Rol y warehouse explícitos: la conexión por llave no trae warehouse por defecto (CURRENT_WAREHOUSE() da None).
sf("USE ROLE ACCOUNTADMIN")       # el administrador de tu cuenta de prueba
sf("USE WAREHOUSE COMPUTE_WH")    # el warehouse X-Small que viene con toda cuenta de prueba
rol, warehouse = sf("SELECT CURRENT_ROLE(), CURRENT_WAREHOUSE()").fetchone()
print("Rol:", rol, "| Warehouse:", warehouse)

sf(f"CREATE SCHEMA IF NOT EXISTS {SF_BASE}.{SF_ESQUEMA}")
sf(f"USE SCHEMA {SF_BASE}.{SF_ESQUEMA}")
sf("ALTER SESSION SET USE_CACHED_RESULT = FALSE")      # medir de verdad, sin devolver resultados guardados

with cronometro("snowflake: despertar el warehouse (SELECT 1)"):
    sf("SELECT 1").fetchall()

sf(f"""
    CREATE OR REPLACE TEMPORARY STAGE viajes_enero
    URL = '{BUCKET_PUBLICO}'
    FILE_FORMAT = (TYPE = PARQUET USE_LOGICAL_TYPE = TRUE USE_VECTORIZED_SCANNER = TRUE)
""")
print("Archivos en el stage:", [(f[0].split("/")[-1], f[1]) for f in sf("LIST @viajes_enero").fetchall()])

CONSULTA_SF = f"""
    SELECT $1:PULocationID::INT                       AS pu_location_id,
           $1:pickup_datetime::TIMESTAMP_NTZ::DATE    AS fecha,
           COUNT(*)                                   AS viajes,
           SUM($1:base_passenger_fare::DOUBLE)        AS ingreso
    FROM @viajes_enero
    WHERE $1:pickup_datetime::TIMESTAMP_NTZ >= '{INICIO}'
      AND $1:pickup_datetime::TIMESTAMP_NTZ <  '{FIN}'
    GROUP BY 1, 2
"""
with cronometro("snowflake: consulta de la pregunta"):
    sf(CONSULTA_SF)
    res_sf = cur.fetch_pandas_all()
res_sf.columns = [c.lower() for c in res_sf.columns]
res_sf["pu_location_id"] = res_sf["pu_location_id"].astype("int64")
print("ID de la consulta (búscala en Snowsight > Monitoring > Query History):", cur.sfqid)
RESULTADOS["snowflake"] = dict(df=res_sf, segundos=TIEMPOS["snowflake: consulta de la pregunta"], pico_mb=pico_mb())
res_sf.head()

**Qué observar.** El tiempo de esta celda no es comparable con el de pandas o Spark «en abstracto»: incluye leer el archivo por la red, y Snowflake trabaja sobre un warehouse del tamaño que tenga tu cuenta. Lo que sí es comparable es que la consulta SQL es casi la misma que escribiste en Spark; cambia la forma de apuntar a la tabla (`@viajes_enero` y las columnas como `$1:nombre`). Si quieres ver cómo se repartió el trabajo, abre la consulta en el *Query Profile* de Snowsight con el ID que imprimió la celda: es la misma lectura de evidencia que hiciste en la clase 3.

### 2.5 Los tres resultados coinciden

Comparar tiempos solo tiene sentido si los tres respondieron lo mismo. La celda siguiente compara, motor contra motor, el número de grupos zona-día, el total de viajes, el total de ingreso (con una tolerancia mínima, porque sumar decimales en distinto orden puede cambiar la última cifra) y, grupo por grupo, cuántos viajes difieren.

In [ ]:
import math

def normalizar(df):
    out = df[["pu_location_id", "fecha", "viajes", "ingreso"]].copy()
    out["pu_location_id"] = out["pu_location_id"].astype("int64")
    out["fecha"] = pd.to_datetime(out["fecha"]).dt.date
    out["viajes"] = out["viajes"].astype("int64")
    out["ingreso"] = out["ingreso"].astype("float64")
    return out.sort_values(["pu_location_id", "fecha"]).reset_index(drop=True)

filas = []
for motor, r in RESULTADOS.items():
    r["norm"] = normalizar(r["df"])
    filas.append(dict(motor=motor, grupos=len(r["norm"]), viajes=int(r["norm"]["viajes"].sum()),
                      ingreso=round(float(r["norm"]["ingreso"].sum()), 2), segundos=round(r["segundos"], 1)))
resumen = pd.DataFrame(filas).set_index("motor")
print(resumen.to_string(formatters={"ingreso": "{:,.2f}".format}))

base = RESULTADOS["pandas"]["norm"]
for motor, r in RESULTADOS.items():
    if motor == "pandas":
        continue
    cruce = base.merge(r["norm"], on=["pu_location_id", "fecha"], how="outer", suffixes=("_pd", "_x"), indicator=True)
    sin_pareja = int((cruce["_merge"] != "both").sum())
    dif_viajes = int((cruce["viajes_pd"] != cruce["viajes_x"]).sum())
    dif_ingreso = float((cruce["ingreso_pd"] - cruce["ingreso_x"]).abs().max())
    ok = (sin_pareja == 0 and dif_viajes == 0 and dif_ingreso < 1e-3
          and math.isclose(resumen.loc["pandas", "ingreso"], resumen.loc[motor, "ingreso"], rel_tol=1e-9))
    print(f"pandas vs {motor}: grupos sin pareja = {sin_pareja}, grupos con viajes distintos = {dif_viajes}, "
          f"mayor diferencia de ingreso = {dif_ingreso:.6f}  ->  {'COINCIDEN' if ok else 'REVISAR'}")

**Qué observar.** La tabla pone lado a lado, para cada motor, cuántos grupos produjo, cuántos viajes sumó, el ingreso total y los segundos que tardó. Las líneas `pandas vs ...` deben decir `COINCIDEN`. Si alguna dice `REVISAR`, no sigas: significa que los motores no respondieron la misma pregunta (un filtro de fechas distinto, una zona nula, un tipo mal convertido) y los tiempos no son comparables. Cuando todo coincide, lee los tiempos con cuidado: pandas gana cuando los datos caben en memoria y la pregunta es corta; Spark empieza a justificar su costo de arranque cuando los datos o el cálculo ya no caben en una máquina; Snowflake paga por leer por la red, pero no necesita que tengas nada instalado ni memoria propia. Ninguno es «el mejor»: es un compromiso entre dónde están los datos, cuánto pesan y quién paga el cómputo.

Cierra la conexión de Snowflake cuando termines de usarla (el warehouse se suspende solo, pero la conexión abierta no hace falta):

In [ ]:
cur.close()
con.close()
print("Conexión a Snowflake cerrada.")

## 3. Spark anota lo que quieres hacer y solo trabaja cuando le pides un resultado

**El problema.** Si cada paso de una transformación se ejecutara al instante, Spark leería 19.6 millones de filas para filtrar, escribiría el resultado intermedio, lo volvería a leer para seleccionar columnas y así sucesivamente. **La idea de ingeniería** es separar *describir* de *ejecutar*. Las *transformaciones* (`where`, `select`, `withColumn`, `groupBy`, `join`) solo agregan un paso al plan y devuelven otro DataFrame al instante. Las *acciones* (`count`, `collect`, `show`, `toPandas`, escribir un archivo) son las que obligan a ejecutar el plan completo. **Lo que resuelve** esa pereza (*lazy evaluation*) es que Spark ve el plan entero antes de tocar un dato, y puede leer solo las columnas que de verdad se usan y empujar los filtros hasta el lector del archivo. **Lo que no resuelve**: un error en la lógica solo aparece cuando ejecutas una acción, no cuando escribes la transformación.

La primera celda construye una cadena de transformaciones y mide cuánto tarda **definirla**. La segunda pide el plan sin ejecutarlo. La tercera ejecuta una acción y mide cuánto tarda **hacerlo**.

In [ ]:
with cronometro("definir: leer"):
    v = spark.read.parquet(str(RUTA_DIR_ENERO))
with cronometro("definir: where"):
    v = v.where(F.col("PULocationID") == 132)
with cronometro("definir: select"):
    v = v.select("pickup_datetime", "base_passenger_fare")
with cronometro("definir: withColumn"):
    v = v.withColumn("tarifa_centavos", F.col("base_passenger_fare") * 100)

print("\nNo se ha leído ninguna fila de datos todavía; solo se anotó el plan.")

In [ ]:
with cronometro("acción: count() ejecuta todo el plan"):
    n = v.count()
print(f"Viajes que salen de la zona 132: {fmt(n)}")

**Qué observar.** Las cuatro líneas de «definir» tardan milisegundos, sin importar que el archivo pese 451 MiB: Spark apenas leyó el pie del archivo Parquet para conocer sus columnas. El tiempo real aparece en el `count()`, que es la acción. Si abres la Spark UI en *Jobs*, ahí está el trabajo de ese `count()`; las transformaciones no generaron ninguno.

### El plan con y sin filtro: lectura de columnas y filtros empujados

`explain()` imprime el plan físico, que se lee de abajo hacia arriba: la última línea es la lectura del archivo (`FileScan parquet`) y las de arriba son lo que se hace con esos datos. En esa línea buscamos dos cosas: `ReadSchema`, las únicas columnas que Spark va a leer del Parquet, y `PushedFilters`, los filtros que se le entregan al lector del archivo para que descarte datos antes de subirlos a memoria. Esa es la ganancia de ser lazy: Spark pudo mirar todo el plan y pedirle al archivo solo lo necesario.

In [ ]:
def texto_plan(df, modo="simple"):
    """El plan de df como texto (explain() imprime; aquí se captura)."""
    salida = StringIO()
    with contextlib.redirect_stdout(salida):
        df.explain(mode=modo)
    return salida.getvalue()


def mostrar_plan(df, claves=None):
    """Imprime el plan; si se dan claves, solo las líneas que contienen alguna de ellas.
    La línea del FileScan es larguísima: se resume a sus filtros empujados y a las columnas que lee."""
    for linea in texto_plan(df).splitlines():
        if claves is not None and not any(c in linea for c in claves):
            continue
        if "FileScan" in linea:
            trozos = [m.group(0) for m in (re.search(r"PushedFilters: \[.*?\]", linea),
                                           re.search(r"ReadSchema: .*$", linea)) if m]
            linea = "FileScan parquet | " + " | ".join(trozos)
        print(linea.strip()[:400])


base_v = spark.read.parquet(str(RUTA_DIR_ENERO))
print("=== Sin filtro: select de 2 columnas ===")
mostrar_plan(base_v.select("PULocationID", "base_passenger_fare"), ["FileScan", "Project"])
print("\n=== Con filtro: where PULocationID = 132 ===")
mostrar_plan(base_v.where(F.col("PULocationID") == 132).select("PULocationID", "base_passenger_fare"),
             ["FileScan", "Filter", "Project"])

**Qué observar.** En las dos salidas, `ReadSchema` lista solo `PULocationID` y `base_passenger_fare`, aunque el archivo tiene 24 columnas: es la poda de columnas. En la segunda, `PushedFilters` incluye `EqualTo(PULocationID,132)`: el filtro viaja hasta el lector Parquet, que puede saltarse grupos de filas cuyas estadísticas descartan el valor. Que un filtro se empuje no garantiza que se salte mucho: depende de cómo estén ordenados los datos dentro del archivo. Prueba cambiar el filtro por otra columna y vuelve a ejecutar la celda para ver cómo cambia `PushedFilters`.

## 4. Qué se paraleliza sin hablar con nadie, qué obliga a juntar filas y qué no escala

### 4.1 Transformaciones estrechas: cada partición trabaja sola

**El problema.** Cuando el trabajo sobre una fila no necesita ninguna otra fila, cada partición se puede procesar sola, sin enviar nada por la red. **La idea de ingeniería** es reconocer esas operaciones: `WHERE`, `SELECT` y `withColumn` son *estrechas* (cada partición de salida sale de una sola partición de entrada). Escalan bien, porque las particiones no se esperan entre sí: con más hilos se procesan más particiones a la vez. **Lo que muestra el plan**: ninguna línea `Exchange`, y el mismo número de particiones antes y después.

In [ ]:
estrecha = base_v.where(F.col("trip_miles") > 0).select("PULocationID", "trip_miles")
print("Particiones antes :", base_v.rdd.getNumPartitions())
print("Particiones después:", estrecha.rdd.getNumPartitions())
print("\nLíneas del plan que importan:")
mostrar_plan(estrecha, ["Exchange", "Filter", "Project", "FileScan"])
print("\n¿Aparece Exchange (movimiento de filas entre particiones)?", "Exchange" in texto_plan(estrecha))

**Qué observar.** El número de particiones no cambió y el plan no tiene `Exchange`. Ese es el rasgo de una transformación estrecha.

### 4.2 Transformaciones anchas: el shuffle

**El problema.** Para contar viajes por zona hay que juntar todas las filas de la misma zona, y esas filas están repartidas en todas las particiones. **La idea de ingeniería** es el *shuffle*: cada partición reparte sus filas por clave (aplica una función hash a la zona) y las envía a la partición que le toca a esa clave. Eso implica escribir datos a disco, moverlos por la red (aquí, entre hilos) y leerlos de nuevo. Es la operación más cara de Spark, y es lo que hacen `GROUP BY`, `JOIN`, `ORDER BY` y `DISTINCT`. **Lo que muestra el plan**: una línea `Exchange` que parte el plan en dos etapas.

In [ ]:
ancha = base_v.groupBy("PULocationID").agg(
    F.count(F.lit(1)).alias("viajes"), F.sum("base_passenger_fare").alias("ingreso"))

print("spark.sql.shuffle.partitions =", spark.conf.get("spark.sql.shuffle.partitions"),
      "(en cuántas particiones se reparte el resultado de un shuffle)")
print("\nPlan completo:")
mostrar_plan(ancha)
with cronometro("groupBy por zona (acción: collect)"):
    filas_zona = ancha.collect()
print("Filas del resultado:", len(filas_zona))

**Qué observar.** El plan se lee de abajo hacia arriba: `FileScan` lee, un `HashAggregate` agrega, un `Exchange hashpartitioning(PULocationID...)` reparte por zona y otro `HashAggregate` junta. Ahora abre la Spark UI, entra a *Jobs*, abre el último trabajo y mira *Stages*: verás **dos etapas**, y la frontera entre ellas es el `Exchange`. La primera etapa escribe datos de shuffle (*Shuffle Write*) y la segunda los lee (*Shuffle Read*). Antes de seguir, la celda siguiente consulta esos números de la Spark UI desde Python, para que no dependas del navegador.

### 4.3 GROUP BY sí se paraleliza: lo hace en dos fases

Puede parecer que agrupar obliga a mover los 19.6 millones de filas. No es así, y la razón es que Spark agrupa en **dos fases**. Primero, cada partición agrega *sus propias* filas (`HashAggregate` con funciones `partial_count`, `partial_sum`): de millones de filas queda una fila por zona y por partición. Solo esas filas parciales viajan por el shuffle. Segundo, después del `Exchange`, un segundo `HashAggregate` (el *final*) suma los parciales de cada zona. Por eso `COUNT`, `SUM`, `MIN` y `MAX` escalan bien. No todas las agregaciones se pueden partir así: calcular una mediana exacta, por ejemplo, necesita ver todos los valores.

In [ ]:
print("Fases de agregación en el plan:")
mostrar_plan(ancha, ["HashAggregate", "Exchange"])


def etapas_recientes(n=3):
    """Etapas ya terminadas, leídas de la API de la Spark UI (la UI vive mientras la sesión esté viva)."""
    import urllib.request
    sc = spark.sparkContext
    url = f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}/stages?status=complete"
    try:
        with urllib.request.urlopen(url, timeout=10) as r:
            etapas = json.load(r)
    except Exception as e:
        print("No se pudo leer la Spark UI desde Python:", type(e).__name__, "- míralas en el navegador.")
        return
    for e in sorted(etapas, key=lambda x: x["stageId"], reverse=True)[:n]:
        print(f"etapa {e['stageId']:>3} | tareas {e.get('numTasks', 0):>4} | "
              f"shuffle escrito: {fmt(e.get('shuffleWriteRecords', 0)):>12} filas ({e.get('shuffleWriteBytes', 0) / MB:,.1f} MB) | "
              f"shuffle leído: {fmt(e.get('shuffleReadRecords', 0)):>12} filas | {e.get('name', '')[:40]}")


print("\nÚltimas etapas ejecutadas (el groupBy fue la más reciente que escribió shuffle):")
etapas_recientes(4)

**Qué observar.** Primero, en el plan: un `HashAggregate` con `partial_count(1)` y `partial_sum(...)` antes del `Exchange`, y otro con `count(1)` y `sum(...)` después. Segundo, en la tabla de etapas: la etapa que escribió shuffle escribió **muy pocas filas comparadas con las 19 663 930 del archivo** (la celda imprime la cifra real). Esa diferencia es el ahorro de la agregación parcial: se movieron los resúmenes de cada partición, no los viajes. Tercero, la etapa que lee el shuffle trabaja con tantas tareas como `spark.sql.shuffle.partitions` (o menos, si Spark la reduce al ver que los datos son pequeños).

### 4.4 Lo que no se paraleliza bien

Hay cuatro situaciones en las que Spark, aunque tenga muchos hilos, termina trabajando casi como una sola máquina. Las cuatro se pueden medir en este notebook.

**a) Una ventana sin `PARTITION BY`.** Una ventana como `ROW_NUMBER() OVER (ORDER BY pickup_datetime)` necesita ordenar *todas* las filas juntas para numerarlas. Sin `PARTITION BY`, Spark no tiene por dónde partir y mueve todo a **una sola partición**, es decir, a una sola tarea. Con `PARTITION BY` zona, cada zona se numera por separado y las zonas se reparten entre las particiones. Para no agotar la memoria del driver, la prueba usa una muestra (5 % de las filas) del archivo.

In [ ]:
FRACCION_VENTANA = 0.05
muestra_v = base_v.select("PULocationID", "pickup_datetime", "base_passenger_fare").sample(False, FRACCION_VENTANA, seed=42)

sin_particion = muestra_v.withColumn("n", F.row_number().over(Window.orderBy("pickup_datetime")))
con_particion = muestra_v.withColumn("n", F.row_number().over(
    Window.partitionBy("PULocationID").orderBy("pickup_datetime")))

print("=== Ventana SIN PARTITION BY ===")
mostrar_plan(sin_particion, ["Exchange", "Window"])
print("\n=== Ventana CON PARTITION BY PULocationID ===")
mostrar_plan(con_particion, ["Exchange", "Window"])

with cronometro("ventana sin PARTITION BY"):
    maximo_1 = sin_particion.agg(F.max("n")).first()[0]
with cronometro("ventana con PARTITION BY zona"):
    maximo_2 = con_particion.agg(F.max("n")).first()[0]
print(f"\nFilas de la muestra: {fmt(muestra_v.count())} | mayor número de fila sin partición: {fmt(maximo_1)} | con partición: {fmt(maximo_2)}")

**Qué observar.** En el plan sin partición aparece `Exchange SinglePartition`: todas las filas viajan a una partición. Spark además imprime en el registro del proceso (no en el notebook) la advertencia `No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation`; es la forma en que Spark te avisa de este problema. En el plan con partición aparece `Exchange hashpartitioning(PULocationID...)`, que sí reparte. Lee los dos tiempos; el efecto crece con los datos, y con 19.6 millones de filas sin partición la única tarea tendría que ordenarlo todo en 2 GB de driver.

**b) Traer todo al driver.** `collect()` y `toPandas()` traen los datos al driver, que es un solo proceso con 2 GB de memoria. Está bien traer un resultado pequeño (el resumen de 262 zonas) y es una mala idea traer los datos crudos. La celda trae un millón de filas de dos columnas, mide cuánta memoria ocupan ya en pandas y las compara con lo que ocuparía traer las 19.6 millones de filas con las 24 columnas (la estimación de la sección 2.2).

In [ ]:
with cronometro("toPandas: 1 millón de filas, 2 columnas"):
    un_millon = base_v.select("PULocationID", "base_passenger_fare").limit(1_000_000).toPandas()
mb_un_millon = un_millon.memory_usage(deep=True).sum() / MB

with cronometro("toPandas: el resumen por zona (resultado del groupBy)"):
    resumen_zonas = ancha.toPandas()
mb_resumen = resumen_zonas.memory_usage(deep=True).sum() / MB

print(f"\n1 millón de filas, 2 columnas: {mb_un_millon:,.1f} MB en el proceso Python | resumen por zona ({len(resumen_zonas)} filas): {mb_resumen:,.3f} MB")
try:
    print(f"Estimación (sección 2.2) de enero completo con todas las columnas: {est_mes:,.0f} MB; el driver de Spark tiene {spark.sparkContext.getConf().get('spark.driver.memory')}.")
except NameError:
    print("Ejecuta la sección 2.2 para ver la estimación de enero completo.")

**Qué observar.** Traer el resumen cuesta casi nada y traer filas crudas cuesta memoria proporcional al número de filas. La regla de ingeniería es que **al driver se traen resultados, no datos**: agrega, filtra o toma una muestra en Spark, y solo entonces `toPandas()`.

**c) Claves desbalanceadas.** Un shuffle reparte por clave, y la clave decide en qué partición cae cada fila. Si una clave concentra una parte enorme de las filas, la tarea que la recibe trabaja mucho más que las demás, y como una etapa termina cuando termina su tarea más lenta, todo el trabajo espera a esa tarea. En los viajes de NYC, la pregunta es qué zonas concentran viajes. La celda mide cuántos viajes sale de cada zona, qué parte del total concentran las primeras y cuántos viajes más tiene la zona más grande que una zona mediana.

In [ ]:
zonas_sp = spark.read.option("header", True).csv(str(RUTA_ZONAS))
zonas_nombre = zonas_sp.select(F.col("LocationID").cast("int").alias("PULocationID"), "Zone", "Borough")

with cronometro("viajes por zona de recogida"):
    por_zona = (base_v.groupBy("PULocationID").count()
                .join(F.broadcast(zonas_nombre), "PULocationID", "left")
                .orderBy(F.desc("count"))
                .toPandas())

total = int(por_zona["count"].sum())
por_zona["porcentaje"] = (100 * por_zona["count"] / total).round(2)
print(por_zona.head(10).to_string(index=False))
mediana = por_zona["count"].median()
print(f"\nZonas con viajes: {len(por_zona)} | viajes de la zona más grande / viajes de la zona mediana = {por_zona['count'].iloc[0] / mediana:.1f}")
print(f"Las 5 zonas más grandes concentran el {100 * por_zona['count'].head(5).sum() / total:.1f} % de los viajes.")
aero = por_zona[por_zona["Zone"].fillna("").str.contains("Airport")]
print("Aeropuertos (posición en el ranking de zonas y % de los viajes):")
print(aero.assign(posicion=aero.index + 1)[["posicion", "Zone", "porcentaje"]].to_string(index=False))

**Qué observar.** Mira cuántas veces más grande es la primera zona que la zona típica, y en qué puestos del ranking aparecen los aeropuertos. Ahora llévalo a las particiones: si repartes los viajes en 200 particiones usando la zona como clave, la partición que recibe una zona grande tendrá muchas más filas que una típica. La celda siguiente lo mide con `spark_partition_id()`, y con eso queda a la vista el trabajo desigual entre tareas. Hay remedios (sal de claves, dividir las claves grandes, o dejar que Spark lo corrija con la ejecución adaptativa), pero primero hay que **medir** el desbalance, que es lo que haces aquí.

In [ ]:
with cronometro("repartir en 200 particiones por zona y contar filas por partición"):
    por_particion_zona = (base_v.repartition(200, "PULocationID")
                          .groupBy(F.spark_partition_id().alias("particion")).count()
                          .toPandas())

con_filas = por_particion_zona[por_particion_zona["count"] > 0]["count"]
print(f"Particiones con filas: {len(con_filas)} de 200")
print(f"Filas de la partición más cargada: {fmt(int(con_filas.max()))} | de la partición mediana: {fmt(int(con_filas.median()))} "
      f"-> la más cargada tiene {con_filas.max() / con_filas.median():.1f} veces más trabajo")

**d) Muchos archivos pequeños.** Cada archivo que Spark escribe lo produce una tarea, y cada archivo que Spark lee cuesta abrirlo, leer su pie y planificarlo. Si escribes con demasiadas particiones, terminas con miles de archivos de unos pocos kilobytes, y las lecturas posteriores pasan más tiempo abriendo archivos que leyendo datos. Lo mismo ocurre en S3 con Athena y Glue, y es la razón por la que `escribir_silver` y `escribir_gold` cuidan cuántas particiones escriben. La celda escribe la misma muestra de 1 % de los viajes con 200 particiones y con 2, cuenta los archivos y mide cuánto tarda leerlos. La carpeta es temporal y se borra al final.

In [ ]:
pequena = base_v.sample(False, 0.01, seed=42).cache()
with cronometro("armar la muestra del 1 %"):
    n_pequena = pequena.count()
print("Filas de la muestra:", fmt(n_pequena))

TMP_ARCHIVOS = DATOS / "tmp_archivos_pequenos"
shutil.rmtree(TMP_ARCHIVOS, ignore_errors=True)
filas_tabla = []
for n_part in (200, 2):
    destino = TMP_ARCHIVOS / f"con_{n_part}_particiones"
    with cronometro(f"escribir con {n_part} particiones"):
        pequena.repartition(n_part).write.mode("overwrite").parquet(str(destino))
    archivos = list(destino.glob("*.parquet"))
    tam_total = sum(a.stat().st_size for a in archivos)
    with cronometro(f"leer y contar ({n_part} particiones)"):
        leidas = spark.read.parquet(str(destino)).count()
    filas_tabla.append(dict(particiones_al_escribir=n_part, archivos=len(archivos),
                            tamano_total_mb=round(tam_total / MB, 1),
                            tamano_medio_kb=round(tam_total / len(archivos) / 1024),
                            filas_leidas=leidas))
pd.DataFrame(filas_tabla)

In [ ]:
pequena.unpersist()
shutil.rmtree(TMP_ARCHIVOS, ignore_errors=True)
print("Carpeta temporal borrada:", not TMP_ARCHIVOS.exists())

**Qué observar.** Las dos filas de la tabla contienen exactamente los mismos datos; solo cambia cuántos archivos los contienen y cuán grande es cada uno. Compara el tamaño medio en kilobytes y los dos tiempos de lectura. La regla práctica de ingeniería es apuntar a archivos de decenas a cientos de megabytes: el número de particiones de salida se controla con `coalesce(n)` (reduce sin shuffle) o `repartition(n)` (reparte con shuffle).

## 5. Broadcast: evitar el shuffle copiando la tabla pequeña

**El problema.** Para saber a qué zona pertenece cada viaje hay que hacer un `JOIN` entre los 19.6 millones de viajes y la tabla de zonas, que tiene 265 filas. Un `JOIN` normal es ancho: para unir por clave, Spark manda **las dos tablas** por un shuffle, y reparte por `LocationID` los 19.6 millones de viajes solo para encontrar el nombre de 265 zonas. **La idea de ingeniería** es notar que una de las dos tablas es diminuta: en lugar de mover los viajes, se copia la tabla de zonas completa a cada ejecutor (*broadcast*), y cada partición de viajes resuelve su `JOIN` localmente, sin mover ninguna fila de viajes. **Lo que resuelve** es eliminar el shuffle del lado grande. **Lo que no resuelve**: si la tabla «pequeña» no cabe en la memoria de cada ejecutor, el broadcast falla o satura la memoria. Por eso Spark solo lo aplica por debajo de un umbral (`spark.sql.autoBroadcastJoinThreshold`, 10 MB por defecto) y por eso `enriquecer` en `silver.py` lo pide explícitamente con `F.broadcast`.

La celda desactiva el broadcast automático para poder comparar de forma justa: primero un `JOIN` normal (Spark elige `SortMergeJoin`) y luego el mismo con `F.broadcast`. Usa una fracción de los viajes para que la prueba no tarde minutos; puedes subir `FRACCION_JOIN` a 1.0 si quieres usarlos todos. Al terminar, restaura la configuración original.

In [ ]:
FRACCION_JOIN = 0.25
umbral_original = spark.conf.get("spark.sql.autoBroadcastJoinThreshold")
viajes_join = base_v.select("PULocationID", "base_passenger_fare").sample(False, FRACCION_JOIN, seed=7)
zonas_join = zonas_sp.select(F.col("LocationID").cast("int").alias("PULocationID"), "Borough", "Zone")

try:
    spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")      # sin broadcast automático
    sin_bc = viajes_join.join(zonas_join, "PULocationID", "left")
    con_bc = viajes_join.join(F.broadcast(zonas_join), "PULocationID", "left")

    print("=== JOIN normal ===")
    mostrar_plan(sin_bc, ["Join", "Exchange", "Sort"])
    print("\n=== JOIN con F.broadcast(zonas) ===")
    mostrar_plan(con_bc, ["Join", "Exchange", "Sort"])

    with cronometro("join SIN broadcast + groupBy borough"):
        r1 = sin_bc.groupBy("Borough").agg(F.count(F.lit(1)).alias("viajes")).orderBy("Borough").collect()
    with cronometro("join CON broadcast + groupBy borough"):
        r2 = con_bc.groupBy("Borough").agg(F.count(F.lit(1)).alias("viajes")).orderBy("Borough").collect()
finally:
    spark.conf.set("spark.sql.autoBroadcastJoinThreshold", umbral_original)

print("\nResultados iguales:", [tuple(x) for x in r1] == [tuple(x) for x in r2])
print("Umbral de broadcast restaurado a:", spark.conf.get("spark.sql.autoBroadcastJoinThreshold"))

**Qué observar.** En el primer plan aparecen `SortMergeJoin`, dos `Exchange hashpartitioning(PULocationID...)` (uno por cada lado, es decir, los viajes también viajan) y dos `Sort`. En el segundo aparecen `BroadcastHashJoin` y un `BroadcastExchange` que mueve solo la tabla de zonas: los viajes no pasan por ningún shuffle. Compara los dos tiempos y confirma que los resultados son iguales: el broadcast cambia *cómo* se calcula, no *qué* se calcula. En la Spark UI, pestaña *SQL / DataFrame*, abre las dos consultas y verás los dos diagramas.

## 6. Lab: escribe las funciones del pipeline

Hasta aquí mediste y comparaste. Ahora escribes la lógica. El pipeline del curso sigue el patrón *medallion*, que ordena los datos en tres capas según cuánto se han transformado:

- **Bronze** guarda cada fuente **tal como llega**: no se modifica, se organiza en una carpeta por fuente y por mes (`fhvhv/mes=2024-01/`, `clima/mes=2024-01/`) y la ingesta se puede repetir sin duplicar nada, porque si el archivo ya existe no se vuelve a descargar. Si algo sale mal más adelante, bronze permite reprocesar sin volver a pedirle los datos a la fuente.
- **Silver** limpia y enriquece: convierte tipos, descarta los viajes imposibles (distancias de cero millas, duraciones de horas o negativas), quita duplicados y une las zonas y el clima.
- **Gold** agrega: una fila por día, zona y empresa, con los números que responden la pregunta.

Lo que escribes aquí es lo mismo que después corre en Airflow en local y luego en Glue, sin reescribirlo: por eso cada función recibe un DataFrame y devuelve un DataFrame (la excepción es la de bronze, que trabaja con archivos), con el mismo nombre de argumentos que las del pipeline del repositorio. Más adelante las copiarás a `src/`.

Antes de escribir, mira cómo es la ingesta real. La celda imprime el código de `bronze.descargar` y los docstrings de las funciones de silver y gold: es lo que tu código tiene que igualar.

In [ ]:
import inspect
from pipeline import silver, gold, io as pio

print(inspect.getsource(bronze_pipeline.descargar))
print("-" * 70)
print(silver.limpiar_viajes.__doc__)
print("-" * 70)
print(silver.enriquecer.__doc__)
print("-" * 70)
print(gold.metricas_diarias_zona.__doc__)

**Qué observar.** `descargar` hace tres cosas, en este orden: comprueba si el destino ya existe (y entonces no hace nada), baja el archivo a una carpeta temporal y solo cuando terminó lo publica en su sitio. Esas tres decisiones son las que vas a reproducir en el ejercicio 1.

### Preparación del lab: una muestra y la forma de las pruebas

Las funciones del lab no corren sobre los 19.6 millones de viajes sino sobre una **muestra determinista del 1 % de enero** (`sample` con semilla 42) repartida en todo el mes. La celda imprime cuántas filas salen exactamente. Así cada prueba tarda segundos y el resultado es el mismo cada vez.

Los datos reales de enero traen muy pocos duplicados (con la clave que usa el lab, `CLAVE_BRUTA`, el mes completo tiene una sola fila repetida entre 19.6 millones), y para que el ejercicio de duplicados tenga algo que encontrar la celda repite **a propósito** una fracción de las filas de la muestra (el 2 %). Esa es la única parte inventada del lab y queda documentada aquí.

La función `verificar` ejecuta la prueba de cada ejercicio e imprime una de tres cosas. `OK` si todo pasó. `PENDIENTE` si todavía queda un hueco sin resolver, que se reconoce por una línea con `...  # TU CÓDIGO AQUÍ`. `REVISAR` seguido de una pista si tu código se ejecutó pero algo no coincide con lo esperado.

In [ ]:
from datetime import date, datetime, timezone
import hashlib

MESES_LAB = ["2024-01"]
FRACCION_MUESTRA = 0.01
CLAVE_BRUTA = ["hvfhs_license_num", "request_datetime", "pickup_datetime",
               "dropoff_datetime", "PULocationID", "DOLocationID"]
CARPETA_BRONZE = DATOS / "bronze_alumno"            # aquí guarda tu ingesta de clima

from pipeline.silver import CARGOS, DISTANCIA_MAX_MI, DURACION_MIN_MIN, DURACION_MAX_MIN, _empresa, clima_diario
from pipeline.gold import COLUMNAS


def descargar_bytes(url, intentos=3):
    """Ya resuelta: descarga una URL y devuelve los bytes tal cual llegan (reintenta si el servidor se satura)."""
    for i in range(1, intentos + 1):
        try:
            with urllib.request.urlopen(url, timeout=60) as r:
                return r.read()
        except Exception:
            if i == intentos:
                raise
            time.sleep(2 * i)


RESULTADO_LAB = {}
HUECO = re.compile(r"(=|^)\s*\.\.\.\s*#\s*TU CÓDIGO AQUÍ", re.M)


def verificar(nombre, funcion, prueba):
    """Corre prueba(funcion) e imprime OK, PENDIENTE o REVISAR con una pista."""
    try:
        fuente = inspect.getsource(funcion)
    except (OSError, TypeError):
        fuente = ""
    if HUECO.search(fuente):
        estado, detalle = "PENDIENTE", "todavía hay un hueco «...  # TU CÓDIGO AQUÍ» sin resolver."
    else:
        try:
            prueba(funcion)
            estado, detalle = "OK", ""
        except AssertionError as e:
            estado, detalle = "REVISAR", str(e)
        except Exception as e:
            estado, detalle = "REVISAR", f"tu función falló con {type(e).__name__}: {str(e)[:300]}"
    RESULTADO_LAB[nombre] = estado
    print(f"{nombre}: {estado}" + (f"  {detalle}" if detalle else ""))

In [ ]:
viajes_lab = spark.read.parquet(str(RUTA_DIR_ENERO))
zonas = spark.read.option("header", True).csv(str(RUTA_ZONAS))

with cronometro("armar la muestra"):
    muestra = viajes_lab.sample(False, FRACCION_MUESTRA, seed=42).cache()
    n_muestra = muestra.count()
repetidas = muestra.sample(False, 0.02, seed=7)
muestra_dup = muestra.unionByName(repetidas)       # la muestra con filas repetidas a propósito
n_dup = muestra_dup.count()
print(f"Muestra: {fmt(n_muestra)} viajes (1 % de enero, semilla 42)")
print(f"Muestra con duplicados añadidos a propósito: {fmt(n_dup)} filas ({fmt(n_dup - n_muestra)} repetidas)")

# Referencia del pipeline del repo sobre la muestra, para que las pruebas de silver y gold
# no dependan de que hayas resuelto otros ejercicios (el clima viene del archivo de pruebas del repo).
clima_pruebas = clima_diario(spark.read.option("multiLine", True).json(str(RAIZ / "tests" / "data" / "open_meteo_sample.json")))
silver_ref = silver.enriquecer(silver.limpiar_viajes(muestra.dropDuplicates(CLAVE_BRUTA), MESES_LAB), zonas, clima_pruebas).cache()
print("Filas de la referencia de silver:", fmt(silver_ref.count()))


def unir_clima(trips: DataFrame, clima: DataFrame) -> DataFrame:
    """Ya resuelta: une el clima del día de recogida (tabla pequeña, en broadcast)."""
    clima_sel = clima.select(F.col("fecha").alias("pickup_date"), "lluvia_mm", "temp_media_c")
    return trips.join(F.broadcast(clima_sel), "pickup_date", "left")

### Ejercicio 1 · Bronze: ingerir el clima de un mes sin repetir la descarga

**Qué hacer.** Escribe `ingerir_clima(mes, base_bronze)`. Debe llamar a Open-Meteo (la URL la arma `url_clima(mes)` del pipeline), guardar la respuesta **sin modificar ni un byte** en `<base_bronze>/clima/mes=AAAA-MM/open_meteo_AAAA-MM.json` (la ruta la arma `pio.ruta_clima`, igual que el pipeline), y dejar al lado un archivo `open_meteo_AAAA-MM.ingesta.json` con los metadatos de la ingesta: de dónde vino (`fuente`), cuándo se descargó (`descargado_en`, en UTC y formato ISO 8601), cuántos bytes tiene (`bytes`) y su huella SHA-256 (`sha256`). Los metadatos van en un archivo aparte precisamente para no tocar el JSON original. Si el archivo ya existe, la función **no vuelve a descargar** y devuelve la misma ruta.

**Por qué importa.** Bronze es la copia fiel de lo que entregó la fuente: si una transformación posterior sale mal, se reprocesa desde bronze sin volver a pedirle nada a la fuente. Por eso el archivo no se modifica, y por eso ejecutar la ingesta dos veces no debe cambiar nada (la ingesta es *idempotente*).

El parámetro `obtener` es la función que descarga los bytes de una URL; por defecto es `descargar_bytes`. Existe para que la prueba pueda comprobar que la segunda ejecución no llama a la red.

In [ ]:
def ingerir_clima(mes: str, base_bronze: Path, obtener=descargar_bytes) -> Path:
    """Descarga el clima de `mes` a bronze (JSON intacto + metadatos) y devuelve la ruta del JSON."""
    destino = Path(pio.ruta_clima(str(base_bronze), mes))
    metadatos = destino.with_name(destino.stem + ".ingesta.json")

    # 1. Si el JSON ya existe no se vuelve a descargar: devuelve `destino`.
    ya_existe = ...  # TU CÓDIGO AQUÍ: ¿existe el archivo?
    if ya_existe:
        return destino

    url = bronze_pipeline.url_clima(mes)
    contenido = obtener(url)                      # bytes tal como los entrega Open-Meteo

    # 2. Crea la carpeta mes=AAAA-MM si no existe y guarda `contenido` SIN modificarlo en `destino`.
    ...  # TU CÓDIGO AQUÍ

    # 3. Escribe en `metadatos` un JSON con: fuente (la url), descargado_en (UTC, ISO 8601),
    #    bytes (len(contenido)) y sha256 (hashlib.sha256(contenido).hexdigest()).
    ...  # TU CÓDIGO AQUÍ

    return destino

In [ ]:
import tempfile


def prueba_e1(f):
    llamadas = []

    def obtener_y_registrar(url):
        llamadas.append(url)
        datos = descargar_bytes(url)
        llamadas.append(datos)
        return datos

    with tempfile.TemporaryDirectory() as tmp:
        base = Path(tmp)
        ruta = f("2024-01", base, obtener=obtener_y_registrar)
        esperada = base / "clima" / "mes=2024-01" / "open_meteo_2024-01.json"
        assert Path(ruta) == esperada, f"La ruta debe ser {esperada}, y devolviste {ruta}."
        assert esperada.exists(), "El JSON no está en la ruta esperada."

        # contenido: JSON válido, intacto y con los 31 días de enero
        contenido = esperada.read_bytes()
        assert len(llamadas) == 2 and contenido == llamadas[1], "El JSON guardado no es idéntico a los bytes descargados: no lo modifiques (ni lo vuelvas a serializar)."
        daily = json.loads(contenido)["daily"]
        assert len(daily["time"]) == 31 and len(daily["precipitation_sum"]) == 31, "Se esperaban 31 días de enero en el bloque «daily»."

        # metadatos
        meta_ruta = esperada.with_name("open_meteo_2024-01.ingesta.json")
        assert meta_ruta.exists(), "Falta el archivo de metadatos open_meteo_2024-01.ingesta.json al lado del JSON."
        meta = json.loads(meta_ruta.read_text())
        faltan = {"fuente", "descargado_en", "bytes", "sha256"} - set(meta)
        assert not faltan, f"A los metadatos les faltan estas claves: {sorted(faltan)}."
        assert meta["fuente"] == llamadas[0], "«fuente» debe ser la URL que se descargó."
        assert meta["bytes"] == len(contenido), "«bytes» no coincide con el tamaño del JSON."
        assert meta["sha256"] == hashlib.sha256(contenido).hexdigest(), "«sha256» no coincide con el contenido del JSON."
        momento = datetime.fromisoformat(meta["descargado_en"])
        assert momento.tzinfo is not None, "«descargado_en» debe incluir zona horaria (UTC)."

        # doble ejecución: no vuelve a descargar ni cambia nada
        antes = (esperada.stat().st_mtime_ns, meta_ruta.read_text())

        def prohibido(url):
            raise AssertionError("La segunda ejecución volvió a llamar a la red: si el archivo ya existe, no debe descargar.")

        f("2024-01", base, obtener=prohibido)
        assert (esperada.stat().st_mtime_ns, meta_ruta.read_text()) == antes, "La segunda ejecución modificó los archivos."


verificar("E1 ingerir_clima", ingerir_clima, prueba_e1)

Ahora ejecuta tu ingesta de verdad, sobre la carpeta bronze del lab. Si todo está bien, la primera ejecución descarga y la segunda no; la celda lo comprueba mirando si la fecha de modificación del archivo cambió. Este archivo es el que usará el resto del lab.

In [ ]:
if RESULTADO_LAB.get("E1 ingerir_clima") != "OK":
    print("Resuelve y verifica el ejercicio 1 (la celda anterior debe decir OK) y vuelve a ejecutar esta celda.")
else:
    ruta_clima_json = ingerir_clima("2024-01", CARPETA_BRONZE)
    t_primera = ruta_clima_json.stat().st_mtime_ns
    ingerir_clima("2024-01", CARPETA_BRONZE)
    print("JSON en bronze:", ruta_clima_json)
    print("La segunda ejecución respetó el archivo:", ruta_clima_json.stat().st_mtime_ns == t_primera)
    print(json.loads(ruta_clima_json.with_name("open_meteo_2024-01.ingesta.json").read_text()))

### Ejercicio 2 · Silver: tipar columnas y descartar los viajes imposibles

**Qué hacer.** Completa `limpiar_tipos(trips)`, que recibe los viajes crudos de bronze y devuelve los viajes limpios con las mismas reglas de `silver.limpiar_viajes`. Hay tres huecos. Primero, `duracion`: en bronze `trip_time` viene en **segundos** y silver trabaja en **minutos**. Segundo, `total_pagado`: lo que pagó el pasajero es la suma de los cargos de la lista `CARGOS` (tarifa base, peajes, `bcf`, impuesto, recargo de congestión y cargo de aeropuerto, donde un nulo cuenta como 0) más la propina. Tercero, `condicion`: se conserva el viaje solo si tiene fecha y zona de recogida, una distancia mayor que 0 y menor que `DISTANCIA_MAX_MI`, una tarifa base mayor que 0 y una duración entre `DURACION_MIN_MIN` y `DURACION_MAX_MIN` (ambos extremos incluidos).

**Por qué importa.** Los datos crudos traen viajes de 0 millas, de horas de duración, sin zona. Si no se descartan, los promedios de gold salen distorsionados y nadie se da cuenta. Esta es una transformación estrecha: cada fila se decide sola, sin mirar a las demás, así que escala sin shuffle. La función `_empresa` (que traduce la licencia a Uber, Lyft, etcétera) ya viene resuelta.

In [ ]:
def limpiar_tipos(trips: DataFrame) -> DataFrame:
    """Tipos, duración, empresa, total pagado y filtros de outliers (mismas reglas que silver.limpiar_viajes)."""
    duracion = ...  # TU CÓDIGO AQUÍ: trip_time (segundos) a minutos, como double
    propina = F.coalesce(F.col("tips").cast("double"), F.lit(0.0))
    total_pagado = ...  # TU CÓDIGO AQUÍ: suma de los CARGOS (nulo = 0, como double) más la propina

    base = trips.select(
        F.to_timestamp("pickup_datetime").alias("pickup_datetime"),
        F.to_timestamp("dropoff_datetime").alias("dropoff_datetime"),
        F.col("PULocationID").cast("int").alias("pu_location_id"),
        F.col("DOLocationID").cast("int").alias("do_location_id"),
        _empresa("hvfhs_license_num").alias("empresa"),
        F.col("trip_miles").cast("double").alias("distancia_mi"),
        duracion.alias("duracion_min"),
        F.col("base_passenger_fare").cast("double").alias("tarifa_base"),
        propina.alias("propina"),
        F.col("driver_pay").cast("double").alias("pago_conductor"),
        total_pagado.alias("total_pagado"),
    )

    condicion = ...  # TU CÓDIGO AQUÍ: la expresión booleana con todas las reglas de la descripción
    return base.where(condicion).withColumn("pickup_date", F.to_date("pickup_datetime"))

In [ ]:
COLS_ENTRADA = (
    "hvfhs_license_num string, pickup_datetime string, dropoff_datetime string, "
    "PULocationID int, DOLocationID int, trip_miles double, trip_time int, "
    "base_passenger_fare double, tolls double, bcf double, sales_tax double, "
    "congestion_surcharge double, airport_fee double, tips double, driver_pay double"
)


def _viaje(pickup="2024-01-10 10:00:00", segundos=1200, pu=43, distancia=3.0, tarifa=12.0, licencia="HV0003"):
    return (licencia, pickup, None, pu, 79, distancia, segundos, tarifa, 0.0, 0.3, 1.0, 2.75, None, 2.0, 9.0)


def prueba_e2(f):
    # 1. un viaje normal sobrevive y trae los valores esperados
    filas = f(spark.createDataFrame([_viaje()], COLS_ENTRADA)).collect()
    assert len(filas) == 1, "Un viaje normal (3 mi, 20 min, tarifa 12) debería sobrevivir y no quedó."
    r = filas[0]
    assert abs(r["duracion_min"] - 20.0) < 1e-9, f"duracion_min debería ser 20.0 (1200 segundos) y es {r['duracion_min']}: trip_time viene en segundos."
    assert r["empresa"] == "Uber", "La empresa de HV0003 debe ser Uber (usa _empresa)."
    esperado = 12.0 + 0.0 + 0.3 + 1.0 + 2.75 + 0.0 + 2.0
    assert abs(r["total_pagado"] - esperado) < 1e-9, f"total_pagado debería ser {esperado} (tarifa + peajes + bcf + impuesto + congestión + aeropuerto con nulo = 0 + propina) y es {r['total_pagado']}."
    assert str(r["pickup_date"]) == "2024-01-10", "pickup_date debe ser la fecha de pickup_datetime."

    # 2. cada tipo de viaje imposible se descarta
    casos = [("distancia 0 mi", dict(distancia=0.0)),
             ("distancia de exactamente 100 mi (el límite es exclusivo)", dict(distancia=100.0)),
             ("tarifa base 0", dict(tarifa=0.0)),
             ("duración de 0 minutos", dict(segundos=0)),
             ("duración de 181 minutos", dict(segundos=181 * 60)),
             ("sin zona de recogida", dict(pu=None))]
    for descripcion, cambios in casos:
        n = f(spark.createDataFrame([_viaje(**cambios)], COLS_ENTRADA)).count()
        assert n == 0, f"No descartaste el viaje con {descripcion}."

    # 3. sobre la muestra real da lo mismo que silver.limpiar_viajes del pipeline
    mia, ref = f(muestra), silver.limpiar_viajes(muestra, MESES_LAB)
    assert mia.columns == ref.columns, f"Las columnas deben ser {ref.columns}, en ese orden; tienes {mia.columns}."
    assert mia.count() == ref.count(), f"Sobre la muestra conservaste {mia.count()} viajes y el pipeline conserva {ref.count()}."
    redondear = lambda d: d.withColumn("total_pagado", F.round("total_pagado", 6))
    assert redondear(mia).exceptAll(redondear(ref)).count() == 0 and redondear(ref).exceptAll(redondear(mia)).count() == 0, \
        "Las filas conservadas coinciden en número pero algún valor es distinto al del pipeline (revisa duración y total_pagado)."


verificar("E2 limpiar_tipos", limpiar_tipos, prueba_e2)

### Ejercicio 3 · Silver: quitar los duplicados por clave

**Qué hacer.** Completa `quitar_duplicados(df, clave)`: debe devolver `df` dejando **una sola fila por cada combinación de valores de las columnas de `clave`**, con las mismas columnas que tenía. Hay dos maneras de hacerlo: `df.dropDuplicates(clave)`, o numerar las filas de cada clave con `row_number()` sobre una ventana `Window.partitionBy(*clave).orderBy(...)` y quedarte con la número 1 (en ese caso recuerda quitar la columna auxiliar).

**Por qué importa, y por qué es un shuffle.** Detectar un duplicado exige comparar una fila con todas las demás de su misma clave, y esas filas pueden estar en cualquier partición. Spark tiene que reunir las filas de cada clave en la misma partición, o sea, un shuffle. Después de la prueba, la celda imprime el plan para que veas el `Exchange`. La clave que usa el lab (`CLAVE_BRUTA`) identifica un viaje por su empresa, la hora de solicitud, la de recogida y la de llegada, y las dos zonas.

In [ ]:
def quitar_duplicados(df: DataFrame, clave: list[str]) -> DataFrame:
    """Una fila por cada combinación de valores de `clave`; conserva las columnas de `df`."""
    resultado = ...  # TU CÓDIGO AQUÍ: dropDuplicates(clave), o row_number() sobre una ventana por clave
    return resultado

In [ ]:
def prueba_e3(f):
    chico = spark.createDataFrame([("a", 1, "x"), ("a", 1, "x"), ("b", 2, "y"), ("a", 1, "x")], "k string, v int, otro string")
    r = f(chico, ["k", "v"])
    assert r.count() == 2, f"De 4 filas con 2 claves distintas deberían quedar 2 y quedaron {r.count()}."
    assert r.columns == chico.columns, f"Debes conservar las columnas {chico.columns} y devolviste {r.columns} (¿dejaste una columna auxiliar?)."

    real = f(muestra_dup, CLAVE_BRUTA)
    esperadas = muestra_dup.select(*CLAVE_BRUTA).distinct().count()
    assert real.count() == esperadas, f"La muestra tiene {esperadas} claves distintas y devolviste {real.count()} filas."
    assert real.select(*CLAVE_BRUTA).distinct().count() == real.count(), "Todavía quedan claves repetidas."
    assert real.columns == muestra_dup.columns, "Las columnas de la salida deben ser las mismas que las de la entrada."


verificar("E3 quitar_duplicados", quitar_duplicados, prueba_e3)

In [ ]:
if RESULTADO_LAB.get("E3 quitar_duplicados") == "OK":
    print(f"Filas con duplicados: {fmt(muestra_dup.count())} -> sin duplicados: {fmt(quitar_duplicados(muestra_dup, CLAVE_BRUTA).count())}")
    print("\nPlan de tu función (busca el Exchange: es el shuffle que junta las filas de la misma clave):")
    mostrar_plan(quitar_duplicados(muestra_dup, CLAVE_BRUTA), ["Exchange", "HashAggregate", "Window", "Sort"])
else:
    print("Resuelve y verifica el ejercicio 3 para ver su plan.")

**Qué observar.** Hay un `Exchange hashpartitioning(...)` sobre las columnas de la clave: es el shuffle que reparte las filas por clave para poder compararlas. Si usaste `dropDuplicates` verás también un `HashAggregate` parcial y uno final, igual que en el `GROUP BY` de la sección 4.3; con la ventana verás `Sort` y `Window`. Pregúntate qué pasaría con un duplicado cuyas dos copias cayeran en particiones distintas si no hubiera shuffle: no se detectaría.

### Ejercicio 4 · Silver: unir las zonas con broadcast

**Qué hacer.** Completa `unir_zonas(trips, zonas)`. Debe unir cada viaje con su zona de recogida (`pu_location_id` contra `LocationID`) con un **`LEFT JOIN`** y pidiendo el **broadcast** de la tabla de zonas, y calcular la columna `zona`: el nombre de la zona, o `Zona <id>` cuando el identificador no está en la tabla. La función ya prepara `zonas_sel` con los nombres de columna que usa silver.

**Por qué importa.** Un `LEFT JOIN` no pierde viajes: los que no encuentran zona se quedan con un nombre genérico en vez de desaparecer, y eso se comprueba con el conteo. El broadcast evita mover los viajes por un shuffle, como viste en la sección 5. La prueba apaga el broadcast automático de Spark para comprobar que **tú** lo pediste explícitamente.

In [ ]:
def unir_zonas(trips: DataFrame, zonas: DataFrame) -> DataFrame:
    """LEFT JOIN con las zonas (broadcast) y columna `zona` (nombre, o 'Zona <id>' si no existe)."""
    zonas_sel = zonas.select(
        F.col("LocationID").cast("int").alias("pu_location_id"),
        F.col("Borough").alias("borough"),
        F.col("Zone").alias("zone_name"),
    )
    unido = ...  # TU CÓDIGO AQUÍ: trips LEFT JOIN zonas_sel por pu_location_id, con zonas_sel en broadcast
    zona = ...  # TU CÓDIGO AQUÍ: zone_name, o 'Zona ' + pu_location_id si zone_name es nulo
    return unido.withColumn("zona", zona).drop("zone_name")

In [ ]:
def prueba_e4(f):
    chicos = spark.createDataFrame([(132, "a"), (9999, "b"), (1, "c")], "pu_location_id int, x string")
    umbral = spark.conf.get("spark.sql.autoBroadcastJoinThreshold")
    try:
        spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")     # sin broadcast automático
        res = f(chicos, zonas)
        plan = texto_plan(res)
        filas = {r["pu_location_id"]: r for r in res.collect()}
    finally:
        spark.conf.set("spark.sql.autoBroadcastJoinThreshold", umbral)
    assert len(filas) == 3, f"Un LEFT JOIN no debe perder viajes: entraron 3 y salieron {len(filas)}."
    assert filas[132]["zona"] == "JFK Airport" and filas[132]["borough"] == "Queens", "La zona 132 debe ser «JFK Airport» del borough «Queens»."
    assert filas[1]["zona"] == "Newark Airport", "La zona 1 debe ser «Newark Airport»."
    assert filas[9999]["zona"] == "Zona 9999", f"Una zona que no está en la tabla debe llamarse «Zona 9999» y se llama {filas[9999]['zona']!r}."
    assert filas[9999]["borough"] is None, "Para una zona inexistente, borough debe quedar nulo."
    assert "zone_name" not in res.columns, "La columna auxiliar zone_name no debe quedar en la salida."
    assert "BroadcastHashJoin" in plan, "El plan no usa BroadcastHashJoin: pide el broadcast de la tabla de zonas con F.broadcast(...)."
    # sobre la muestra: no se pierde ningún viaje
    limpio = silver.limpiar_viajes(muestra, MESES_LAB)
    assert f(limpio, zonas).count() == limpio.count(), "Sobre la muestra, la unión cambió el número de viajes."


verificar("E4 unir_zonas", unir_zonas, prueba_e4)

### Ejercicio 5 · Gold: agregar por día, zona y empresa, y por qué los promedios se ponderan

**Qué hacer, primera parte.** Completa `agregar_gold(silver)`: una fila por `fecha` (la `pickup_date`), `zona` y `empresa`, con `viajes` (cuántas filas), `ingreso_total` (suma de `total_pagado`, redondeada a 2 decimales), `propina_promedio` y `distancia_promedio` (promedios de `propina` y `distancia_mi`, redondeados a 2 decimales) y `lluvia_mm` (la precipitación del día, que es la misma para todos los viajes de ese día: `max`). Las columnas de salida son las de `COLUMNAS`, en ese orden. Es lo que hace `gold.metricas_diarias_zona`.

**Por qué importa.** Gold es lo que consume Snowflake: una tabla pequeña, con un grano claro (la clave `fecha, zona, empresa` no se repite) y sin que nadie vuelva a tocar los viajes individuales. Es una transformación ancha: un `groupBy`, con sus dos fases.

**Qué hacer, segunda parte.** Un día querrás una tabla más gruesa, por fecha y zona sin separar por empresa, y la tentación es promediar los promedios de gold. Eso es un error: si Uber hizo 8 viajes con propina media de 1 dólar y Lyft hizo 2 con propina media de 2, la propina media de la zona no es 1.5 sino `(8·1 + 2·2) / 10 = 1.2`. Un promedio de promedios trata igual a una empresa con 2 viajes que a otra con 8. La forma correcta es **ponderar por viajes**. Completa `resumen_por_zona(gold)`, que agrupa un gold por `fecha` y `zona` y calcula `propina_promedio` y `distancia_promedio` ponderando por `viajes`, y suma `viajes` e `ingreso_total`. `lluvia_mm` sigue siendo `max`. La prueba compara tu resultado con el promedio calculado directamente desde los viajes.

In [ ]:
def agregar_gold(silver: DataFrame) -> DataFrame:
    """Métricas diarias por zona de recogida y empresa (mismo grano y columnas que gold.metricas_diarias_zona)."""
    agregado = ...  # TU CÓDIGO AQUÍ: groupBy(fecha, zona, empresa).agg(viajes, ingreso_total, propina_promedio, distancia_promedio, lluvia_mm)
    return agregado.select(*COLUMNAS)


def resumen_por_zona(gold: DataFrame) -> DataFrame:
    """Gold por fecha y zona (sin separar por empresa), con promedios ponderados por viajes."""
    viajes = F.sum("viajes")
    propina_ponderada = ...  # TU CÓDIGO AQUÍ: suma de propina_promedio * viajes, entre el total de viajes, redondeada a 2
    distancia_ponderada = ...  # TU CÓDIGO AQUÍ: lo mismo con distancia_promedio
    return gold.groupBy("fecha", "zona").agg(
        viajes.alias("viajes"),
        F.round(F.sum("ingreso_total"), 2).alias("ingreso_total"),
        propina_ponderada.alias("propina_promedio"),
        distancia_ponderada.alias("distancia_promedio"),
        F.max("lluvia_mm").alias("lluvia_mm"),
    )

In [ ]:
ESQUEMA_SILVER = ("pickup_date date, zona string, empresa string, total_pagado double, propina double, "
                  "distancia_mi double, lluvia_mm double")
ESQUEMA_GOLD = ("fecha date, zona string, empresa string, viajes long, ingreso_total double, "
                "propina_promedio double, distancia_promedio double, lluvia_mm double")


def prueba_e5(f):
    chico = spark.createDataFrame([
        (date(2024, 1, 1), "A", "Uber", 10.0, 1.0, 2.0, 3.5),
        (date(2024, 1, 1), "A", "Uber", 20.0, 3.0, 4.0, 3.5),
        (date(2024, 1, 1), "A", "Lyft", 8.0, 0.0, 1.0, 3.5),
        (date(2024, 1, 1), "B", "Uber", 5.0, 0.0, 1.0, 3.5),
        (date(2024, 1, 2), "A", "Uber", 7.0, 0.5, 6.0, None),
    ], ESQUEMA_SILVER)
    g = f(chico)
    assert g.columns == COLUMNAS, f"Las columnas deben ser {COLUMNAS}; tienes {g.columns}."
    filas = {(r["fecha"], r["zona"], r["empresa"]): r for r in g.collect()}
    assert len(filas) == 4, f"Con estos datos deben salir 4 filas (una por fecha, zona y empresa) y salieron {len(filas)}."
    a1 = filas[(date(2024, 1, 1), "A", "Uber")]
    assert a1["viajes"] == 2, "Dos viajes de Uber en la zona A el día 1: viajes debe ser 2."
    assert a1["ingreso_total"] == 30.0, "ingreso_total es la suma de total_pagado (10 + 20)."
    assert a1["propina_promedio"] == 2.0 and a1["distancia_promedio"] == 3.0, "Los promedios de propina y distancia de Uber en A deben ser 2.0 y 3.0."
    assert a1["lluvia_mm"] == 3.5, "lluvia_mm es la lluvia del día."
    assert filas[(date(2024, 1, 2), "A", "Uber")]["lluvia_mm"] is None, "Si el día no tiene dato de lluvia, lluvia_mm debe quedar nulo."
    # sobre la referencia de silver: igual que gold.metricas_diarias_zona
    mia, ref = f(silver_ref), gold.metricas_diarias_zona(silver_ref)
    assert mia.exceptAll(ref).count() == 0 and ref.exceptAll(mia).count() == 0, "Sobre la referencia de silver, tu resultado difiere del de gold.metricas_diarias_zona."


def prueba_e5_ponderado(f):
    chico = spark.createDataFrame([
        (date(2024, 1, 1), "A", "Uber", 8, 40.0, 1.0, 3.0, 3.5),
        (date(2024, 1, 1), "A", "Lyft", 2, 30.0, 2.0, 8.0, 3.5),
    ], ESQUEMA_GOLD)
    r = f(chico).collect()
    assert len(r) == 1, "Las dos filas (Uber y Lyft) deben juntarse en una sola por fecha y zona."
    r = r[0]
    assert r["viajes"] == 10 and r["ingreso_total"] == 70.0, "viajes e ingreso_total se suman: 10 y 70.0."
    assert abs(r["propina_promedio"] - 1.2) < 1e-9, f"propina_promedio debería ser (8·1 + 2·2) / 10 = 1.2 y es {r['propina_promedio']}: pondera por viajes, no promedies los promedios."
    assert abs(r["distancia_promedio"] - 4.0) < 1e-9, f"distancia_promedio debería ser (8·3 + 2·8) / 10 = 4.0 y es {r['distancia_promedio']}."
    # contra el promedio calculado directamente desde los viajes
    mio = f(gold.metricas_diarias_zona(silver_ref))
    directo = silver_ref.groupBy(F.col("pickup_date").alias("fecha"), "zona").agg(
        F.count(F.lit(1)).alias("n"), F.avg("propina").alias("p"), F.avg("distancia_mi").alias("d"))
    cruce = mio.join(directo, ["fecha", "zona"])
    assert cruce.count() == directo.count() == mio.count(), "Deben salir exactamente las combinaciones fecha-zona de silver."
    malos = cruce.where((F.col("viajes") != F.col("n"))
                        | (F.abs(F.col("propina_promedio") - F.col("p")) > 0.011)
                        | (F.abs(F.col("distancia_promedio") - F.col("d")) > 0.011)).count()
    assert malos == 0, f"En {malos} combinaciones fecha-zona el promedio no coincide con el calculado directo desde los viajes (tolerancia de redondeo: 0.011)."


verificar("E5a agregar_gold", agregar_gold, prueba_e5)
verificar("E5b resumen_por_zona", resumen_por_zona, prueba_e5_ponderado)

### Tu pipeline completo contra el del repositorio

Si los cinco ejercicios dicen `OK`, ya puedes encadenar tus funciones en el mismo orden que usa el pipeline: bronze (tu ingesta del clima), duplicados, limpieza de tipos, zonas, clima y agregación a gold. La celda siguiente las encadena sobre la muestra con duplicados y compara el resultado, fila por fila, con el que produce el código del repositorio (`silver.limpiar_viajes`, `silver.enriquecer`, `gold.metricas_diarias_zona`) sobre los mismos datos y el mismo clima. Si coinciden, tus funciones hacen lo mismo que las del pipeline y se pueden llevar a `src/`.

In [ ]:
pendientes = [k for k, v in RESULTADO_LAB.items() if v != "OK"]
if len(RESULTADO_LAB) < 6 or pendientes:
    print("Todavía no: resuelve y verifica los ejercicios", pendientes or "(faltan ejecutar celdas)")
else:
    ruta_clima_json = ingerir_clima("2024-01", CARPETA_BRONZE)
    clima_lab = clima_diario(spark.read.option("multiLine", True).json(str(ruta_clima_json)))

    with cronometro("tu pipeline: duplicados -> tipos -> zonas -> clima -> gold"):
        silver_lab = unir_clima(unir_zonas(limpiar_tipos(quitar_duplicados(muestra_dup, CLAVE_BRUTA)), zonas), clima_lab)
        gold_lab = agregar_gold(silver_lab).cache()
        n_gold = gold_lab.count()

    referencia = gold.metricas_diarias_zona(
        silver.enriquecer(silver.limpiar_viajes(muestra.dropDuplicates(CLAVE_BRUTA), MESES_LAB), zonas, clima_lab))
    iguales = gold_lab.exceptAll(referencia).count() == 0 and referencia.exceptAll(gold_lab).count() == 0
    print(f"Filas de tu gold: {fmt(n_gold)} | viajes: {fmt(gold_lab.agg(F.sum('viajes')).first()[0])}")
    print("Tu gold es idéntico al del pipeline del repo:", iguales)

In [ ]:
if "gold_lab" in globals():
    diario = (gold_lab.groupBy("fecha").agg(F.sum("viajes").alias("viajes"), F.max("lluvia_mm").alias("lluvia_mm"))
              .orderBy("fecha").toPandas())
    print(diario.to_string(index=False))
    print("\nCorrelación día a día entre lluvia (mm) y viajes de la muestra:", round(float(diario["viajes"].corr(diario["lluvia_mm"])), 3))
else:
    print("Primero termina la celda anterior.")

**Qué observar.** Esta tabla es el primer vistazo a la pregunta del curso, con tus propias funciones. No es una conclusión: es **un mes**, **una muestra del 1 %** y una correlación entre dos series de 31 puntos, sin controlar nada más (el día de la semana pesa mucho en los viajes y se mezcla con la lluvia). Sirve para comprobar que la tubería de datos funciona de punta a punta; contestar la pregunta de verdad es lo que harás en el Taller 2, con el año completo y con cuidado.

### Cuándo cerrar Spark

La sección 7 corre el pipeline desde la terminal, y los tests de `make test` también arrancan su propio Spark. En esta máquina solo cabe un proceso de Spark a la vez, así que **antes de pasar a la terminal debes cerrar el de este notebook** con la celda siguiente. Después de ejecutarla, las celdas que usan `spark` dejan de funcionar hasta que reinicies el kernel y las vuelvas a correr desde la sección 0.5. Si quieres seguir experimentando en el notebook, deja esta celda para el final.

In [ ]:
muestra.unpersist()
silver_ref.unpersist()
spark.stop()
print("Spark cerrado. Ya puedes usar la terminal: make test, pipeline.run, etc.")

## 7. «A mano primero»: el mismo pipeline desde la terminal, y luego con `make`

Todo lo que hiciste en este notebook corre también como un programa, sin notebook. La regla de trabajo del curso es la misma cada vez que aparece una herramienta: primero el comando completo escrito a mano, con cada parte explicada; después el mismo comando con otro paso; y solo entonces el atajo.

### Paso 1. El comando completo

Abre una terminal en el Code Editor (menú *Terminal > New Terminal*) y ejecuta:

```bash
cd ~/bigdata && PYTHONPATH=src .venv/bin/python -m pipeline.run --step silver --env local --engine local --mes 2024-01
```

Parte por parte:

- `cd ~/bigdata` entra a la carpeta del repositorio. Todos los comandos de esta sección se ejecutan desde ahí; en otra carpeta no encontrarán ni `src` ni `.venv`.
- `PYTHONPATH=src` le dice a Python que busque paquetes en la carpeta `src`. El paquete `pipeline` vive ahí y no está instalado en el entorno, así que sin esto aparece `ModuleNotFoundError: No module named 'pipeline'`. (Es la misma ruta que agregaste en la celda 0.2 con `sys.path.insert`.)
- `.venv/bin/python` es el Python del proyecto, el que tiene `pyspark` instalado. Escribir la ruta completa evita usar por error otro Python.
- `-m pipeline.run` ejecuta el módulo `src/pipeline/run.py` como programa. Es el punto de entrada de todos los pasos.
- `--step silver` elige el paso: `silver` limpia y enriquece (tus ejercicios 2, 3 y 4 hacen esto mismo). Los otros valores son `gold`, `all`, y `bronze` o `muestra`, que solo corren en la nube.
- `--env local` elige el archivo de configuración `src/pipeline/config/local.yaml`: lee la muestra de `./lake/muestra` y escribe en `./lake/silver`.
- `--engine local` elige el motor: Spark `local[*]` con 2 GB de driver, la misma sesión que arrancaste en la celda 0.5.
- `--mes 2024-01` procesa solo ese mes.

**La primera vez** el programa necesita copiar la muestra desde tu bucket a `./lake/muestra`, y para eso necesita saber cuál es tu bucket. Si ves `No hay muestra ... falta LAKE_BUCKET`, agrega al final del comando `--lake-bucket <tu bucket>`; la celda siguiente imprime el comando listo con tu bucket. Las siguientes veces no hace falta.

In [ ]:
print(f"cd ~/bigdata && PYTHONPATH=src .venv/bin/python -m pipeline.run --step silver --env local --engine local --mes 2024-01 --lake-bucket {BUCKET}")

**Salida esperada.** Al ejecutarlo verás líneas de registro como estas, con tus números en lugar de los puntos suspensivos (lo verás al ejecutar):

```text
... INFO Paso=silver env=local engine=local meses=2024-01
... INFO Lectura viajes: ... filas (archivos ... MB, límite ninguno) desde .../lake/muestra, meses 2024-01
... INFO Limpieza: ... -> ... filas (... descartadas)
... INFO Calidad OK [silver]: ... filas
```

Léelas como una historia: cuántas filas entraron, cuántas sobrevivieron a la limpieza (la misma lógica que tu ejercicio 2) y que las reglas de calidad pasaron. Si una regla de calidad falla, el programa termina con un `QualityError` que dice la capa, la columna y cuántas filas están mal. Si Java no está disponible, el error aparece al arrancar Spark; si queda un Spark abierto en el notebook, verás que el programa se queda esperando o que falla por memoria: ciérralo con la celda de la sección 6.

**Ojo con el tamaño:** en el entorno `local`, el pipeline no lee el archivo de enero completo que usaste en el notebook, sino la **muestra** que dejó `make muestra` en `~/bigdata/lake/muestra` (unos 8 mil viajes por mes). Por eso tarda segundos. La línea `Lectura viajes: ... desde .../lake/muestra` lo confirma. El mes completo se procesa en Glue, en la clase 7.


### Paso 2. Repetirlo con otro paso

Cambia solo el paso y vuelve a ejecutar. `gold` lee lo que escribió silver y escribe las métricas diarias por zona (tus ejercicios 5a y 5b hacen esto):

```bash
cd ~/bigdata && PYTHONPATH=src .venv/bin/python -m pipeline.run --step gold --env local --engine local --mes 2024-01
```

También puedes comprobar el código sin ejecutar el pipeline completo, con las pruebas y el verificador de estilo, siempre desde `~/bigdata`:

```bash
cd ~/bigdata && .venv/bin/python -m pytest tests/test_silver.py
cd ~/bigdata && .venv/bin/ruff check src
```

`pytest` ejecuta las pruebas de silver sobre datos de ejemplo (por eso también arranca Spark: no lo lances con el notebook abierto y encendido) y debe terminar con `passed`. `ruff check` revisa que el código siga las reglas de estilo y debe imprimir `All checks passed!`.

### Paso 3. Ahora, `make`

Abre el archivo `Makefile` de la raíz del repositorio. Cada *target* es un nombre para un comando que ya sabes escribir. Para comprobarlo, `make -n` muestra lo que se ejecutaría **sin ejecutarlo**:

```bash
cd ~/bigdata && make -n run-local STEP=silver MES=2024-01
```

Salida:

```text
STEP=silver MES=2024-01 LIMITE= bash scripts/run-local.sh
```

El target `run-local` solo pasa tus variables al script `scripts/run-local.sh`, y ese script arma los argumentos (`--step silver --env local --engine local --mes 2024-01`), calcula el bucket del lake si hace falta y ejecuta exactamente el `python -m pipeline.run` que escribiste a mano, con `PYTHONPATH` ya puesto. Ahora sí, el atajo:

```bash
cd ~/bigdata && make run-local STEP=silver MES=2024-01
```

Y lo mismo con las pruebas y el estilo: `make -n test` muestra `bash scripts/test.sh`, que ejecuta `ruff check`, `ruff format --check` y `pytest`, es decir, los comandos del paso 2 y un poco más. Se ejecuta con `cd ~/bigdata && make test`. Recuerda: **cierra Spark en el notebook antes** (la celda de la sección 6), porque `make test` y `make run-local` arrancan el suyo.

| Error común | Causa | Cómo evitarlo |
|---|---|---|
| `No rule to make target 'run-local'` | Ejecutaste `make` fuera de `~/bigdata` | Empieza siempre con `cd ~/bigdata &&` |
| `ModuleNotFoundError: No module named 'pipeline'` | Falta `PYTHONPATH=src` en el comando a mano | Escríbelo antes de `.venv/bin/python`, o usa `make` |
| `No hay muestra ... falta LAKE_BUCKET` | Primera ejecución sin bucket | Agrega `--lake-bucket <tu bucket>` al comando a mano; con `make` se calcula solo |
| El programa falla al arrancar Spark o se queda esperando | El notebook sigue teniendo Spark abierto | Ejecuta la celda `spark.stop()` de la sección 6 o reinicia el kernel |